<a href="https://colab.research.google.com/github/noobylub/Dissertation_Project/blob/master/test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Check what device I am running on, and I want to make sure I am running on TPU
import torch

def check_gpu():
    """Check GPU availability"""
    if torch.cuda.is_available():
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
        return torch.device("cuda")


    else:
        print("📱 Using CPU")
        return torch.device("cpu")

device = check_gpu()

✅ GPU: Tesla T4


# **Loading English Dataset**
In this section, we load the English Dataset for vector extraction later on

In [ ]:
# Loading GoEmotion Dataset and the module used to analyse them
!wget -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv
!pip install pandas

In [ ]:
# Preliminary analysis
import pandas as pd
# Analysing from google colab
data_path = "/content/data/full_dataset/goemotions_2.csv"
df = pd.read_csv(data_path)
for contents in df.columns:
  print(contents)

In [54]:
anger_statements = [row_data['text'] for index, row_data in df.iterrows() if row_data['anger'] == 1]
happiness_statements = [row_data['text'] for index, row_data in df.iterrows() if row_data['joy'] == 1]

In [55]:
for anger_statement in anger_statements:
  print("Anger Statement", anger_statement)


Anger Statement Shhh don't give them the idea!
Anger Statement That really sucks. How old is your baby?
Anger Statement I think I'm kinda the same way like I change up so much everything I say feels like a lie... See just did it again. Fuck.
Anger Statement I don’t like dogs so I don’t care.
Anger Statement Hey [NAME]! **TRY DRAWING TWO CARDS NOW, YOU STUPID FUCKING BASTARD!!**
Anger Statement All these women are going to have a tough time supporting each other while they explain to the American public why their competitors are unqualified
Anger Statement Upvoted twice because this is EXACTLY what they think it is and it sucks rat balls.
Anger Statement I got the same call, voicemail was definitely Chinese but I deleted it and marked the number as Spam/Blocked 
Anger Statement > It's crazy you don't understand this. i r o n y
Anger Statement Be careful with this. This is the same rationale lots of people used to hate on the Covington students.
Anger Statement What the fuckkkkk burn itt

In [ ]:
for happiness_statement in happiness_statements:
  print("Happiness Statement", happiness_statement)

# **Loading Indonesian Dataset**
In this section, we load the Indonesian Dataset for vector extraction later on

In [47]:
import pandas as pd
# Analysing from google colab
anger_path = "/content/AngerData.csv"
happiness_path = "/content/JoyData.csv"
anger_df = pd.read_csv(anger_path, sep='\t')
happiness_df = pd.read_csv(happiness_path, sep='\t')

for contents in anger_df.columns:
  print(contents)
for contents in happiness_df.columns:
  print(contents)

Tweet
Label
Tweet
Label


In [49]:
anger_statements_id = [row_data['Tweet'] for index, row_data in anger_df.iterrows() ]
happiness_statements_id = [row_data['Tweet'] for index, row_data in happiness_df.iterrows() ]

In [ ]:
for idx, happiness_statement in enumerate(happiness_statements_id):
  print("Happiness Statement at", idx, happiness_statement)

In [ ]:
for idx,anger_statement in enumerate(anger_statements_id):
  print("Anger Statement at " ,idx, anger_statement)

# **HuggingFace and Model Setup**

In [ ]:
# If running on google colab website
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

In [ ]:
from dotenv import load_dotenv
import os

# Load the .env file (adjust path if needed)
load_dotenv('.env')

# Now access the variables
hf_token = os.getenv('HF_TOKEN')
print(hf_token)

In [ ]:
# Install bitsandbytes for quantisation
!pip install -U bitsandbytes>=0.46.1

Use these models: https://huggingface.co/google/gemma-7b
<br/>
Second model to test: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct

In [ ]:
# Lading the transformer models into the GPU

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from dotenv import load_dotenv
# import os

# # Load the .env file (adjust path if needed)
# load_dotenv('.env')

# # Now access the variables
# hf_token = os.getenv('HF_TOKEN')

# Quantisation
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

# Trying to run inference on Llama 3.1 model
# Add device mapping and quantization
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_id,
    token=hf_token,
    padding_side='left',


)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    token=hf_token,
    device_map="auto",           # Auto device placement
    quantization_config=bnb_config
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [ ]:
import torch

def get_model_size(model):
    param_size = 0
    buffer_size = 0

    for param in model.parameters():
        param_size += param.nelement() * param.element_size()

    for buffer in model.buffers():
        buffer_size += buffer.nelement() * buffer.element_size()

    total_size = (param_size + buffer_size) / 1024**3  # GB

    print(f"Parameters: {param_size / 1024**3:.2f} GB")
    print(f"Buffers: {buffer_size / 1024**3:.2f} GB")
    print(f"Total: {total_size:.2f} GB")

    return total_size

# Check your model size
model_size_gb = get_model_size(model)

Parameters: 8.46 GB
Buffers: 0.00 GB
Total: 8.46 GB


In [ ]:
def hook_extract_vector(module, input, output):
    if isinstance(output, tuple):
        hidden_state = output[0]
    else:
        hidden_state = output  # Fixed: was output.last_hidden_state

    extracted_activations.append(hidden_state.detach().cpu())
    return output

print("Hook function redefined!")

Hook function redefined!


In [ ]:
model.model._forward_hooks.clear()

# **Vector Extraction**
We mostly follow this method: https://elib.dlr.de/218629/1/The_Effectiveness_of_Style_Vectors_for_Steering_Large_Language_Models_A_Human_Evaluation.pdf
<br/>
One critical aspect to note is we pass forward pass and extract the activaiton from that forward pass


In [ ]:
# This is for extracting vectors
# We mostly follow this method: https://elib.dlr.de/218629/1/The_Effectiveness_of_Style_Vectors_for_Steering_Large_Language_Models_A_Human_Evaluation.pdf
# Extracting the input representation and averaging them to determine layer representation

import torch


# model generates with the given prompt and extracts the vectors
def generate_extract(user_text: str, model, tokenizer, max_new_tokens=200, layer_idx=15):

    messages = [
        {"role": "user", "content": user_text},

    ]

    encoded_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():
        outputs = model(
            **encoded_inputs,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            output_hidden_states = True,
        )
        # output = model(**inputs)
        # outputs = output

    # input_length = input_ids.shape[1]
    # generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    # print(generated_text)

    # extract_hook.remove()

    return outputs.hidden_states

In [ ]:
# This would be for steering
import torch

# Code set up beforehand

# Global storage (define outside functions)
# extracted_activations = []

# Method to extract vectors from model
def extract_vector(module, input, output,vector_steer,strength):
    # The output from model.model (LlamaModel) during generation is typically a BaseModelOutputWithPast object.
    # We want to extract the last_hidden_state tensor from it.
    if isinstance(output, tuple):
        hidden_state = output[0]
    else:
        hidden_state = output

    # Detach the tensor from the graph and move to CPU to save memory if not needed for backprop
    # extracted_activations.append(hidden_state.detach().cpu())

    # IMPORTANT: Hooks should return the original output or a modified one. If only observing, return original output.
    # If steerign you can modify it first and return that
    return hidden_state + (vector_steer * strength )

# model generates with the given prompt and extracts the vectors
def generate_steer(user_text: str, system_text:str, model, tokenizer, max_new_tokens=200, layer_idx=15):
    extracted_activations.clear()

    extract_hook = model.model.layers[layer_idx].register_forward_hook(extract_vector)
    print("Attached to layer", layer_idx)

    messages = [
        {"role": "user", "content": user_text},
        {"role":"system", "content":system_text}
    ]

    encoded_inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(model.device)

    input_ids = encoded_inputs["input_ids"]
    attention_mask = encoded_inputs["attention_mask"]

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    input_length = input_ids.shape[1]
    generated_text = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    print(generated_text)

    extract_hook.remove()

    return generated_text, extracted_activations.copy()

In [ ]:
# Loading dataset, do this for colab
!wget -P data/full_dataset/ https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_2.csv

--2026-03-16 00:36:24--  https://storage.googleapis.com/gresearch/goemotions/data/full_dataset/goemotions_1.csv
Resolving storage.googleapis.com (storage.googleapis.com)... 74.125.200.207, 74.125.130.207, 74.125.68.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|74.125.200.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14174600 (14M) [application/octet-stream]
Saving to: ‘data/full_dataset/goemotions_1.csv’

goemotions_1.csv    100%[===================>]  13.52M  6.13MB/s    in 2.2s    

2026-03-16 00:36:26 (6.13 MB/s) - ‘data/full_dataset/goemotions_1.csv’ saved [14174600/14174600]



In [ ]:
# Load the dataset

In [ ]:
# Prompts designed to ellicity joy emotion
prompt = "in more than 100 words describe how your day is going"


In [ ]:
generated = generate_extract(prompt, model, tokenizer, max_new_tokens=200, layer_idx=13)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [ ]:

vector = (generated)
# print(text)
for i in range(len(vector)):
  print(vector[i].shape)



torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])
torch.Size([1, 112, 4096])


In [ ]:
print(vectors2[1].shape)

torch.Size([1, 1, 4096])
